# Phase 2 agentic (LangGraph) vs Phase 1 — Amazon Reviews 2023 demo

**Phase 1:** one forward pass — **LLaMA-3-8B + your LoRA**, review **text only** (same prompts as `llama_sentiment_baseline_train.ipynb`).

**Phase 2 (implemented demo):** a small **multi-step graph** (LangGraph) with the same backbone model:
1. **Analyst** — text-only review analysis (hypothesis **H** + draft rating).
2. **RAG Prover** — *toy grounding*: product **title + details** from metadata (stand-in for LanceDB retrieval).
3. **Visual Verifier** — **BLIP** image caption as **Ev** (stand-in for a vision auditor).
4. **Critic** — second LLM call that reconciles H with **Gf** + **Ev** and outputs the **final** `Sentiment (1-5)`.

This is a **faithful skeleton** of the architecture in `Agentic_Sentiment_LLaMA3.html` §§12–13, not production LanceDB/LLaVA. **N** is small — use for demos; metrics are **not** statistically significant.

**Prerequisite:** trained adapter from `llama_sentiment_baseline_train.ipynb` (`output_final.zip` or `./output/final`).

### Google Colab

1. **Runtime → Change runtime type → GPU** (T4 / L4 recommended; CPU will be very slow and may OOM on LLaMA).
2. Upload **`output_final.zip`** to **`/content/`** (Files pane), or unzip so **`/content/output/final`** exists.
3. **Llama-3** is gated: create a [HF token](https://huggingface.co/settings/tokens), accept the model card, then either:
   - Colab **Secrets** (key `HF_TOKEN`) — the next cell logs in automatically, or
   - Run `huggingface-cli login` in a terminal cell.
4. If **`bitsandbytes`** errors after install, use **Runtime → Restart session**, then **run all** from the top.


## 1. Install


In [ ]:
%pip install -q datasets transformers accelerate peft safetensors sentencepiece huggingface_hub
%pip install -q "bitsandbytes>=0.43.0"
%pip install -q pandas pillow requests matplotlib timm
%pip install -q langgraph


## 1b. Colab: GPU check + Hugging Face login (Llama-3)


In [ ]:
import os
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

from huggingface_hub import login

_tok = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
if not _tok:
    try:
        from google.colab import userdata

        _tok = userdata.get("HF_TOKEN")
    except Exception:
        _tok = None
if _tok:
    login(token=_tok, add_to_git_credential=False)
    print("HF: logged in from token.")
else:
    print(
        "HF: no token found. Add Colab Secret HF_TOKEN or set HF_TOKEN env; else run: !huggingface-cli login"
    )


## 2. Configuration


In [ ]:
import os
import random

try:
    import google.colab  # noqa: F401

    _IN_COLAB = True
except ImportError:
    _IN_COLAB = False

# Colab: cwd is usually /content — put zip here or use Drive copy
if _IN_COLAB:
    _root = "/content"
    ADAPTER_ZIP = os.path.join(_root, "output_final.zip")
    ADAPTER_PATH = os.environ.get("ADAPTER_PATH", os.path.join(_root, "output", "final"))
else:
    ADAPTER_ZIP = os.environ.get("ADAPTER_ZIP", "output_final.zip")
    ADAPTER_PATH = os.environ.get("ADAPTER_PATH", "./output/final")

BASE_MODEL = "meta-llama/Meta-Llama-3-8B"

REVIEW_CONFIG = "raw_review_All_Beauty"
META_CONFIG = "raw_meta_All_Beauty"
N_SAMPLES = 12
SEED = 42

MAX_NEW_TOKENS_ANALYST = 128
MAX_NEW_TOKENS_CRITIC = 160
POOL_SIZE = 5
PROMPT_MAX_LENGTH = 1024

BLIP_MODEL = "Salesforce/blip-image-captioning-base"
MAX_IMAGE_BYTES = 2_000_000

random.seed(SEED)
print("Colab:", _IN_COLAB, "| ADAPTER_ZIP:", ADAPTER_ZIP, "| ADAPTER_PATH:", ADAPTER_PATH)


## 3. Load Amazon Reviews 2023 (reviews + metadata + image URLs)


In [ ]:
import re
import warnings
from typing import Any, Dict, List, Optional

import pandas as pd
import requests
from datasets import load_dataset
from PIL import Image
from io import BytesIO

warnings.filterwarnings("ignore")

print("Loading reviews:", REVIEW_CONFIG)
rev = load_dataset("McAuley-Lab/Amazon-Reviews-2023", REVIEW_CONFIG, trust_remote_code=True)
rev_split = rev["full"]
n = min(N_SAMPLES, len(rev_split))
indices = random.sample(range(len(rev_split)), n) if len(rev_split) >= n else list(range(len(rev_split)))
rows = [rev_split[int(i)] for i in indices]

print("Loading metadata:", META_CONFIG)
meta_ds = load_dataset("McAuley-Lab/Amazon-Reviews-2023", META_CONFIG, split="full", trust_remote_code=True)
needed_asins = {str(r.get("parent_asin") or r.get("asin") or "") for r in rows}
needed_asins.discard("")
meta_by_asin = {}
MAX_META_SCAN = 200_000
if needed_asins:
    for i in range(min(len(meta_ds), MAX_META_SCAN)):
        if len(meta_by_asin) >= len(needed_asins):
            break
        r = meta_ds[i]
        pa = str(r.get("parent_asin") or r.get("asin") or "")
        if pa in needed_asins and pa not in meta_by_asin:
            meta_by_asin[pa] = r

def first_image_url(meta: dict) -> Optional[str]:
    im = meta.get("images") or {}
    if isinstance(im, dict):
        for k in ("hi_res", "large", "thumb"):
            for u in im.get(k) or []:
                if u and isinstance(u, str) and u.startswith("http"):
                    return u
    return None

def safe_details(meta: dict) -> str:
    d = meta.get("details")
    if d is None:
        return ""
    if isinstance(d, str):
        return d[:800]
    return str(d)[:800]

records: List[Dict[str, Any]] = []
for row in rows:
    pa = str(row.get("parent_asin") or row.get("asin") or "")
    meta = meta_by_asin.get(pa, {})
    title = (row.get("title") or meta.get("title") or "").strip()
    text = (row.get("text") or "").strip()
    rating = row.get("rating")
    try:
        gt = int(round(float(rating)))
    except (TypeError, ValueError):
        gt = 3
    gt = max(1, min(5, gt))
    cat = (meta.get("main_category") or REVIEW_CONFIG.replace("raw_review_", "").replace("_", " ") or "Unknown")
    records.append({
        "parent_asin": pa,
        "review_text": text[:4000],
        "ground_truth_stars": gt,
        "main_category": cat,
        "meta_title": (meta.get("title") or title)[:500],
        "details": safe_details(meta),
        "image_url": first_image_url(meta),
    })

df_raw = pd.DataFrame(records)
print(df_raw[["ground_truth_stars", "main_category"]].head())
print("Rows:", len(df_raw))


## 4. BLIP captions (Visual Verifier input **Ev**)

Frees GPU memory before loading LLaMA.


In [ ]:
caption_by_idx: Dict[int, str] = {}
import torch
from transformers import BlipProcessor, BlipForConditionalGeneration

device_blip = "cuda" if torch.cuda.is_available() else "cpu"
print("BLIP device:", device_blip)
proc = BlipProcessor.from_pretrained(BLIP_MODEL)
blip = BlipForConditionalGeneration.from_pretrained(BLIP_MODEL).to(device_blip)
blip.eval()

def caption_from_url(url: str) -> str:
    try:
        r = requests.get(url, timeout=15, headers={"User-Agent": "Mozilla/5.0"})
        r.raise_for_status()
        if len(r.content) > MAX_IMAGE_BYTES:
            return ""
        img = Image.open(BytesIO(r.content)).convert("RGB")
        inputs = proc(images=img, return_tensors="pt").to(device_blip)
        out = blip.generate(**inputs, max_new_tokens=40)
        return proc.tokenizer.decode(out[0], skip_special_tokens=True).strip()
    except Exception as e:
        return f"[image unavailable: {e}]"

for i, row in df_raw.iterrows():
    u = row.get("image_url")
    caption_by_idx[i] = caption_from_url(u) if isinstance(u, str) and u.startswith("http") else ""

del blip, proc
if torch.cuda.is_available():
    torch.cuda.empty_cache()

df_raw["image_caption"] = [caption_by_idx.get(i, "") for i in range(len(df_raw))]
print(df_raw[["image_caption"]].head(3))


## 5. Prompt helpers (Phase 1 = baseline notebook style)


In [ ]:
from dataclasses import dataclass
from typing import Optional


SENTIMENT_INSTRUCTION = (
    "Evaluate the sentiment expressed in user reviews and classify each one according to its sentiment rating. "
    "Use a five-point scale: 1-2 negative, 3 neutral, 4-5 positive."
)
RATING_DESCRIPTIONS = {
    1: "Comments show a high level of dissatisfaction and negativity (rating 1).",
    2: "Comments show dissatisfaction (rating 2).",
    3: "Comments are mixed or neutral (rating 3).",
    4: "Comments show satisfaction (rating 4).",
    5: "Comments show strong satisfaction and positivity (rating 5).",
}


@dataclass
class DataConfig:
    use_cot: bool = True
    cot_phrase: str = "Let's take it one step at a time."
    use_one_shot: bool = True


data_cfg = DataConfig()


def format_one_shot(review, rating, cfg: DataConfig):
    desc = RATING_DESCRIPTIONS.get(rating, f"Rating {rating}.")
    return f"Review: {review}\nSentiment (1-5): {rating}. {desc}"


def build_prompt_phase1(review: str, cfg: DataConfig, one_shot_example: Optional[str] = None) -> str:
    parts = [SENTIMENT_INSTRUCTION]
    if cfg.use_cot:
        parts.append(cfg.cot_phrase)
    if one_shot_example and cfg.use_one_shot:
        parts.append("\n\nExample:\n" + one_shot_example)
    parts.append("\n\nReview to classify:\n" + review)
    parts.append(
        '\nAfter your reasoning, end with exactly one line starting with "Sentiment (1-5):" '
        "followed by the rating 1–5 and a short justification (paper-style output)."
    )
    parts.append("\nSentiment (1-5):")
    return "\n".join(parts)


def create_one_shot_pool(samples, cfg: DataConfig, pool_size: int = 5):
    by_rating = {r: [] for r in range(1, 6)}
    for s in samples:
        r = s.get("ground_truth_stars") or s.get("rating")
        if r in by_rating:
            by_rating[r].append(s)
    pool = []
    for r in range(1, 6):
        if by_rating[r]:
            pool.append(random.choice(by_rating[r]))
    if not pool and samples:
        pool = [samples[0]]
    return pool[:pool_size]


def extract_rating_from_output(text: str) -> int:
    text = (text or "").strip()
    if not text:
        return 3
    low = text.lower()
    key = "sentiment (1-5)"
    if key in low:
        idx = low.rfind(key)
        tail = text[idx : idx + 500]
        m = re.search(r"[Ss]entiment\s*\(1-5\)\s*[:=]\s*([1-5])", tail)
        if m:
            return int(m.group(1))
        m = re.search(r"[:=]\s*([1-5])\b", tail)
        if m:
            return int(m.group(1))
    m = re.search(r"[Rr]ating\s*[:=]\s*([1-5])\b", text)
    if m:
        return int(m.group(1))
    for line in reversed(text.splitlines()):
        line = line.strip()
        m = re.match(r"^([1-5])\s*[\.\:\)]", line)
        if m:
            return int(m.group(1))
        m = re.match(r"^([1-5])$", line)
        if m:
            return int(m.group(1))
    m2 = re.search(r"\b([1-5])\b", text)
    if m2:
        return int(m2.group(1))
    digits = re.findall(r"[1-5]", text)
    if digits:
        return int(digits[-1])
    return 3


sample_dicts = df_raw.to_dict("records")
one_shot_pool = create_one_shot_pool(sample_dicts, data_cfg, pool_size=POOL_SIZE)


def pick_one_shot_for_row(row: dict, pool) -> str:
    gt = int(row["ground_truth_stars"])
    candidates = [p for p in pool if int(p["ground_truth_stars"]) != gt]
    if not candidates:
        candidates = pool
    ex = random.choice(candidates)
    return format_one_shot(ex["review_text"], int(ex["ground_truth_stars"]), data_cfg)


## 6. Load LLaMA + PEFT


In [ ]:
import zipfile

_extract_root = "/content" if _IN_COLAB else "."
_zip_candidates = []
for z in (ADAPTER_ZIP, "output_final.zip", "/content/output_final.zip", "./output_final.zip"):
    if z and z not in _zip_candidates:
        _zip_candidates.append(z)
_unzipped = False
for zpath in _zip_candidates:
    if zpath and os.path.isfile(zpath):
        with zipfile.ZipFile(zpath) as z:
            z.extractall(_extract_root)
        print("Unzipped:", zpath, "->", _extract_root)
        _unzipped = True
        break
if not _unzipped:
    print("No zip found; tried:", _zip_candidates)
    print("If adapter is already on disk, ensure ADAPTER_PATH exists.")

if not os.path.isdir(ADAPTER_PATH):
    raise FileNotFoundError(
        f"Adapter not found at {ADAPTER_PATH}. On Colab: upload output_final.zip to /content/ or set ADAPTER_PATH to your unzipped folder."
    )

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

try:
    import bitsandbytes  # noqa: F401
    use_4bit = True
except Exception:
    use_4bit = False

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model_kwargs = {"trust_remote_code": True, "device_map": "auto", "torch_dtype": torch.float16}
if use_4bit:
    model_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
    )

model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **model_kwargs)
model = PeftModel.from_pretrained(model, ADAPTER_PATH)
model.eval()
print("Model ready (4-bit)" if use_4bit else "Model ready (fp16)")


_gen_device = getattr(model, "device", None) or next(model.parameters()).device


@torch.inference_mode()
def generate_text(prompt: str, max_new_tokens: int) -> str:
    inputs = tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=PROMPT_MAX_LENGTH
    ).to(_gen_device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True)


## 7. LangGraph: Analyst → RAG Prover → Visual Verifier → Critic

- **RAG Prover:** formats metadata as a fixed "retrieved spec" string (**Gf**).
- **Visual Verifier:** passes BLIP caption as **Ev** (no extra LLM).
- **Critic:** one reconciliation call with the same LoRA model.


In [ ]:
from typing import TypedDict

from langgraph.graph import END, StateGraph

try:
    from langgraph.graph import START as _LG_START

    _USE_START_NODE = True
except ImportError:
    _LG_START = None
    _USE_START_NODE = False


class AgentState(TypedDict, total=False):
    row: dict
    one_shot: str
    analyst_output: str
    analyst_rating: int
    spec_grounding: str
    visual_evidence: str
    critic_output: str
    final_rating: int


def node_analyst(state: AgentState) -> AgentState:
    row = state["row"]
    prompt = build_prompt_phase1(row["review_text"], data_cfg, one_shot_example=state["one_shot"])
    out = generate_text(prompt, MAX_NEW_TOKENS_ANALYST)
    return {
        "analyst_output": out,
        "analyst_rating": extract_rating_from_output(out),
    }


def node_rag_prover(state: AgentState) -> AgentState:
    row = state["row"]
    gf = (
        "Retrieved product specification grounding (Gf) — simulate LanceDB / catalog hit:\n"
        f"- Category: {row.get('main_category', '')}\n"
        f"- Title: {str(row.get('meta_title', ''))[:300]}\n"
        f"- Details: {str(row.get('details', ''))[:500]}"
    )
    return {"spec_grounding": gf}


def node_visual_verifier(state: AgentState) -> AgentState:
    row = state["row"]
    cap = (row.get("image_caption") or "").strip() or "(no image caption — treat visual evidence as missing)"
    ev = f"Visual evidence (Ev) — BLIP image caption (hardware auditor stand-in):\n{cap[:600]}"
    return {"visual_evidence": ev}


def build_critic_prompt(state: AgentState) -> str:
    row = state["row"]
    ao = (state.get("analyst_output") or "")[:900]
    return (
        "You are the Critic agent in a multi-step sentiment pipeline. Reconcile the Analyst's hypothesis "
        "with factual product grounding (Gf) and visual evidence (Ev). "
        "If the review clearly conflicts with the product identity or visible product type, adjust the final "
        "star rating toward a more appropriate score; if evidence is weak or missing, trust the Analyst more.\n\n"
        f"{state.get('visual_evidence', '')}\n\n"
        f"{state.get('spec_grounding', '')}\n\n"
        "Analyst preliminary output (text-only on the review):\n"
        f"{ao}\n\n"
        "Review to finalize:\n"
        f"{row['review_text'][:2500]}\n\n"
        'Reason step by step, then end with exactly one line starting with "Sentiment (1-5):" '
        "with the final integer 1–5 and a brief justification."
    )


def node_critic(state: AgentState) -> AgentState:
    prompt = build_critic_prompt(state)
    # Shorter context budget for critic if prompt is huge
    out = generate_text(prompt, MAX_NEW_TOKENS_CRITIC)
    return {"critic_output": out, "final_rating": extract_rating_from_output(out)}


def build_graph():
    g = StateGraph(AgentState)
    g.add_node("analyst", node_analyst)
    g.add_node("rag_prover", node_rag_prover)
    g.add_node("visual_verifier", node_visual_verifier)
    g.add_node("critic", node_critic)
    if _USE_START_NODE:
        g.add_edge(_LG_START, "analyst")
    else:
        g.set_entry_point("analyst")
    g.add_edge("analyst", "rag_prover")
    g.add_edge("rag_prover", "visual_verifier")
    g.add_edge("visual_verifier", "critic")
    g.add_edge("critic", END)
    return g.compile()


agent_app = build_graph()
print("LangGraph compiled: analyst → rag_prover → visual_verifier → critic")


## 8. Run Phase 1 vs Phase 2 on the same rows


In [ ]:
results = []
for _, row in df_raw.iterrows():
    r = row.to_dict()
    one = pick_one_shot_for_row(r, one_shot_pool)

    p1_prompt = build_prompt_phase1(r["review_text"], data_cfg, one_shot_example=one)
    p1_out = generate_text(p1_prompt, MAX_NEW_TOKENS_ANALYST)
    phase1_rating = extract_rating_from_output(p1_out)

    s2 = agent_app.invoke(
        {"row": r, "one_shot": one},
    )
    phase2_rating = int(s2["final_rating"])

    results.append({
        "gt": int(r["ground_truth_stars"]),
        "phase1": phase1_rating,
        "phase2": phase2_rating,
        "analyst": int(s2["analyst_rating"]),
        "snippet": (r["review_text"] or "")[:70].replace("\n", " "),
    })

df_res = pd.DataFrame(results)
print(df_res.to_string(index=False))

def acc(pred, gt):
    return (pred == gt).mean() if len(gt) else 0.0

def mae(pred, gt):
    return (pred - gt).abs().mean() if len(gt) else 0.0

a1, a2 = acc(df_res["phase1"], df_res["gt"]), acc(df_res["phase2"], df_res["gt"])
print(f"\nAccuracy Phase 1 (single pass): {a1:.3f}")
print(f"Accuracy Phase 2 (graph):     {a2:.3f}")
print(f"MAE Phase 1: {float(mae(df_res['phase1'], df_res['gt'])):.3f} | MAE Phase 2: {float(mae(df_res['phase2'], df_res['gt'])):.3f}")
print("\nNote: Phase 2 is not guaranteed to beat Phase 1 on every slice; on some samples grounding helps.")


## 9. Plot


In [ ]:
import matplotlib.pyplot as plt

x = range(len(df_res))
plt.figure(figsize=(11, 4))
plt.plot(x, df_res["gt"], "ko-", label="Ground truth")
plt.plot(x, df_res["phase1"], "s-", label="Phase 1 (1× LLM)")
plt.plot(x, df_res["phase2"], "^-", label="Phase 2 (Analyst+RAG+Vis+Critic)")
plt.xticks(x, [f"s{i}" for i in x])
plt.ylabel("Stars (1–5)")
plt.xlabel("Sample")
plt.legend()
plt.title("Amazon Reviews 2023 demo — same LoRA, Phase 1 vs agentic Phase 2")
plt.tight_layout()
plt.show()


## 10. What is / isn’t implemented

| Piece | Status |
|--------|--------|
| LangGraph orchestration | Yes |
| Analyst + Critic (2 LLM calls in graph + 1 for Phase 1 compare) | Yes — Phase 1 uses 1 call; Phase 2 graph uses Analyst + Critic (RAG/Vis nodes are non-LLM) |
| LanceDB + CLIP index | No — metadata string = **toy Gf** |
| LLaVA spatial verifier | No — **BLIP caption = toy Ev** |
| Reflection loop | No — single critic pass (add a conditional edge later) |
